In [6]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [8]:
# ===============================
# 1. Đọc dữ liệu review từ URL
# ===============================
review_url = "https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw/review_categories/Health_and_Personal_Care.jsonl"
review_df = pd.read_json(review_url, lines=True)
print(review_df.head())
# ===============================
# 2. Lọc user có >=5 reviews
# ===============================
user_counts = review_df['user_id'].value_counts()
valid_users = user_counts[user_counts >= 5].index
review_filtered = review_df[review_df['user_id'].isin(valid_users)]

   rating                                              title  \
0       4  12 mg is 12 on the periodic table people! Mg f...   
1       5                 Save the lanet using less plastic.   
2       5                                          Fantastic   
3       4  It holds the water and makes bubbles.  That's ...   
4       1                                         Not for me   

                                                text images        asin  \
0  This review is more to clarify someone else’s ...     []  B07TDSJZMR   
1  Love these easy multitasking bleach tablets. B...     []  B08637FWWF   
2  I have been suffering a couple months with hee...     []  B07KJVGNN5   
3  It's cheap and it does what I wanted.  The "ma...     []  B007HY7GC2   
4  Didn't do a thing for me. Not saying they don'...     []  B08KYJLF5T   

  parent_asin                       user_id               timestamp  \
0  B07TDSJZMR  AFKZENTNBQ7A7V7UXW5JJI6UGRYQ 2020-02-06 00:49:35.902   
1  B08637FWWF  AEVWAM3

In [9]:
# ===============================
# 3. Đọc dữ liệu metadata từ URL
# ===============================
meta_url = "https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw/meta_categories/meta_Health_and_Personal_Care.jsonl"
meta_df = pd.read_json(meta_url, lines=True)
print(meta_df.head())

unique_asins = review_filtered['parent_asin'].unique()
meta_filtered = meta_df[meta_df['parent_asin'].isin(unique_asins)].copy()


3. Đang đọc dữ liệu metadata...
            main_category                                              title  \
0  Health & Personal Care  Silicone Bath Body Brush Exfoliator Shower Bac...   
1  Health & Personal Care  iPhone 7 Plus 8 Plus Screen Protector, ZHXIN T...   
2  Health & Personal Care  Zig Zag Rolling Machine 70mm Size With FREE BO...   
3  Health & Personal Care    Sting-Kill Disposable Wipes 8 Each ( Pack of 5)   
4  Health & Personal Care  Heated Eyelash Curler Mini Portable Electric E...   

   average_rating  rating_number  \
0             3.9              7   
1             3.8              2   
2             3.9              7   
3             4.1              6   
4             3.3              8   

                                            features  \
0                                                 []   
1  [Tough and Robust: Like all 78X screen protect...   
2                                                 []   
3                                             

In [10]:
# ===============================
# 4. Join hai bảng
# ===============================
final_df = review_filtered.merge(meta_filtered, on='parent_asin', how='left')

In [ ]:

# ===============================
# 5. Chia dữ liệu thành 3 tập (train, validation, test)
# ===============================
# Chia lần 1: tách 30% để làm validation + test
train_df, temp_df = train_test_split(final_df, test_size=0.3, random_state=42)
# Chia lần 2: từ 30% tách thành 2 nửa (15% val, 15% test)
valid_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)
# ===============================
# Lưu vào thư mục data
# ===============================
os.makedirs('data', exist_ok=True)

train_df.to_parquet('..data/train.parquet')
valid_df.to_parquet('..data/valid.parquet')
test_df.to_parquet('..data/test.parquet')
# Lưu thêm bảng đã join đầy đủ
final_df.to_parquet('..data/final_data.parquet')